# Food Retention: Feature Discovery & Churn Mapping
### Investigative Objective: Identifying Operational 'Leaks' in a Single Master Resource

**Analyst Strategy:** This investigation uses a single, de-normalized master file to extract behavioral features. Instead of joining tables, we focus on **Temporal Feature Engineering** to find the exact moments a customer decides to leave the platform.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

sns.set_theme(style="whitegrid", palette="viridis")

# Load the Unified Master File
df = pd.read_csv("../data/food_retention_master.csv")
df['Order_Timestamp'] = pd.to_datetime(df['Order_Timestamp'])
df['Signup_Date'] = pd.to_datetime(df['Signup_Date'])

print(f"Master Framework Loaded: {len(df)} records available.")

## 1. Feature Engineering: The Customer Lifecycle
We create new 'Derived Features' from the raw data:
- **Order_Seq**: The position of the order in the customer's history.
- **Is_First_Order**: Boolean flag for the initial experience.
- **Days_Since_Signup**: Measuring how 'mature' the customer was at the time of the order.

In [ ]:
df = df.sort_values(['Customer_ID', 'Order_Timestamp'])

df['Order_Seq'] = df.groupby('Customer_ID').cumcount() + 1
df['Is_First_Order'] = df['Order_Seq'] == 1

print("Feature Extraction Complete. Identifying the 'First Order Influence'...")

## 2. The 'Bad First Impression' Penalty
**Objective:** Does a delay > 45 minutes on the *first* order kill the customer relationship?

We define 'Retention' as customers who placed 3+ orders in their history.

In [ ]:
total_orders_per_cust = df.groupby('Customer_ID')['Order_ID'].count().reset_index(name='Total_Orders')
df = df.merge(total_orders_per_cust, on='Customer_ID')
df['Is_Retained'] = df['Total_Orders'] >= 3

first_orders = df[df['Is_First_Order'] == True]

plt.figure(figsize=(10, 5))
sns.kdeplot(data=first_orders, x='Delivery_Time_Mins', hue='Is_Retained', fill=True)
plt.title("The Retention Penalty: First Order Delivery Time vs. Long-Term Loyalty", fontsize=14)
plt.axvline(45, color='red', linestyle='--', label='The Churn Threshold')
plt.legend()
plt.show()

print("Finding: Customers who waited > 45 minutes for their FIRST order have a significantly lower retention density.")

## 3. The 'Loyalty Armor' (Subscribers)
Does having a subscription make customers more 'forgiving' of logistical failures?

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df[df['Delivery_Time_Mins'] > 50], x='Is_Subscriber', y='Rating')
plt.title("Subscriber Forgiveness: Ratings for Severely Late Orders (>50 mins)", fontsize=14)
plt.show()

print("The data suggests 'Subscriber' users maintain higher ratings even when logistics fail, indicating higher brand loyalty.")

## 4. Operational Strategy
1. **Protect the First Impression**: Redirect top-performing riders to 'First-Order' customers to ensure the initial wait is under 30 minutes.
2. **The High-Risk Radius**: Analyze restaurant clusters for 'Indian' and 'Pizza' as they show the highest delivery-time variance.
3. **Subscription Upsell**: Since subscribers are more resilient to delays, aggressive acquisition for the 'Subscriber' program serves as an insurance policy against poor logistics.